Cell 1 — Imports

In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

Cell 2 — Define Input Paths

In [19]:
ASSETS_PATH = (
    "/kaggle/input/datasets/kamyarpm/deep-image-restoration-assets01"
)

CORRUPTED_IMAGE_PATH = (
    f"{ASSETS_PATH}//images/corrupted_image.jpg"
)

ORIGINAL_IMAGE_PATH = (
    f"{ASSETS_PATH}/images/original_image.jpg"
)

DENOISING_MODEL_PATH = (
    f"{ASSETS_PATH}/models/denoising_model.h5"
)

Cell 3 — Check Files

In [20]:
print(
    "Corrupted image exists:",
    os.path.exists(CORRUPTED_IMAGE_PATH)
)

print(
    "Original image exists:",
    os.path.exists(ORIGINAL_IMAGE_PATH)
)

print(
    "Denoising model exists:",
    os.path.exists(DENOISING_MODEL_PATH)
)

Corrupted image exists: True
Original image exists: True
Denoising model exists: True


Cell 4 — Create Output Directory

In [22]:
PHASE_1_OUTPUT_DIR = (
    "/kaggle/working/phase_1_results"
)

os.makedirs(
    PHASE_1_OUTPUT_DIR,
    exist_ok=True
)

DENOISED_IMAGE_PATH = (
    f"{PHASE_1_OUTPUT_DIR}/phase1_denoised.jpg"
)

Cell 5 — Denoising Function

In [24]:
def denoising(
    image_path,
    dest_path,
    model_path
):

    # Load the pre-trained denoising model
    model = tf.keras.models.load_model(
        model_path,
        compile=False
    )

    # Read the input image
    image = cv2.imread(
        image_path
    )

    # Convert BGR to RGB
    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    # Resize the image to the model input size
    image = cv2.resize(
        image,
        (256, 256)
    )

    # Normalize pixel values to the range [0, 1]
    input_tensor = (
        image.astype(np.float32) / 255.0
    )

    # Add the batch dimension
    input_tensor = np.expand_dims(
        input_tensor,
        axis=0
    )

    # Run model inference
    denoised_tensor = model.predict(
        input_tensor,
        verbose=0
    )

    # Convert the output back to the range [0, 255]
    denoised_image = (
        denoised_tensor[0] * 255.0
    )

    # Clip pixel values to a valid image range
    denoised_image = np.clip(
        denoised_image,
        0,
        255
    ).astype(np.uint8)

    # Convert RGB back to BGR
    denoised_image = cv2.cvtColor(
        denoised_image,
        cv2.COLOR_RGB2BGR
    )

    # Save the denoised image
    cv2.imwrite(
        dest_path,
        denoised_image
    )

    print(
        f"Denoised image saved to: {dest_path}"
    )

Cell 6 — Run Phase 1

In [25]:
denoising(
    image_path=CORRUPTED_IMAGE_PATH,
    dest_path=DENOISED_IMAGE_PATH,
    model_path=DENOISING_MODEL_PATH
)

I0000 00:00:1784500884.011620      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784500884.015098      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Denoised image saved to: /kaggle/working/phase_1_results/phase1_denoised.jpg


I0000 00:00:1784500887.097900     149 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
